# Fingerprint Embedding Network — sequence ANN backend `(B, L, 2, 23, 128)`

Speaker verification on **sequences** of triplet-STDP weight fingerprints — one frame
per piece from `training/be_ann_w_fingerprints/prepare_fingerprints_piecewise.ipynb`
(48-ish frames for a ~5s clip, variable per clip). CNN frame encoder trained with
AAMSoftmax (ArcFace) closed-set classification on **VoxCeleb2** shards, evaluated
open-set (cosine similarity) on disjoint **VoxCeleb1** speakers — same training/eval
scaffolding as `ann_backend.ipynb`, but the model now reads genuine frame-to-frame
(piece-to-piece) relationships via a masked ECAPA-TDNN-style stack over the piece
axis, instead of treating each clip as one static image.

**What changed vs. `ann_backend.ipynb`** (that notebook trained on the OLD whole-clip,
192-dim/3-types-per-channel fingerprints — a single static image per wav):
- Geometry: `(3,23,192)` → `(2,23,128)` per frame (matches `train.py`'s current
  128-dim/2-types-per-channel architecture — see
  `training/tonotopic_plasticity_bound/train.py`).
- One clip = a **variable-length sequence** of frames (one per piece), not one static
  fingerprint. Batching pads to each batch's own max length + a boolean mask.
- Model: the old spatial `AttentiveStatsPool` (which pooled a flattened *spatial*
  remnant of the tonotopic-offset/hidden-neuron axes, not real time) is retired in
  favor of a per-frame 2D CNN + global-average-pool, feeding a masked ECAPA-TDNN
  stack (`Bottle2neck` x3, dilation 2/3/4, ported from `ecapa_baseline.ipynb`) +
  masked attentive statistics pooling over the **piece** axis — genuine
  frame-to-frame temporal structure.
- `PKSampler`, `AAMSoftmax`, `retrieval_metrics`/EER eval, the AMP/optimizer/scheduler/
  checkpointing training loop, and the `NEUTRAL_FILL` ablation mechanism are reused
  UNCHANGED from `ann_backend.ipynb` — they only ever touch the final `(N,256)`
  embedding or operate independently of the frame axis.
- **Data loading: memory-mapped, not eager.** At full-corpus scale this dataset is
  ~95GB (vs. ~4.4GB for the old whole-clip fingerprints), which does not fit in
  Kaggle RAM. `load_split` keeps the big frame arrays on disk as `np.memmap`s (one
  per shard) and only reads the specific (sample, frame-range) slice each batch
  actually needs; per-sample metadata (`person_ids`/`record_ids`/`labels`/`n_pieces`)
  is still loaded eagerly (negligible size). `PKSampler` is deliberately left drawing
  from the FULL dev class set every batch (unchanged) rather than switched to
  shard-restricted sampling — that was already tried upstream and reverted because it
  starved the AAM-softmax head (a shard doesn't contain every speaker); mmap avoids
  that regression while still fitting in RAM. This requires shards to be saved as a
  DIRECTORY of plain `.npy` files rather than one `.npz` — see
  `prepare_fingerprints_piecewise.ipynb`'s intro for why (`.npz` doesn't actually
  support lazy mmap in numpy, verified empirically).

Set the three ablation flags at the top of the config cell. `NEUTRAL_FILL` keeps input
channel count / param count identical across all 7 valid flag combinations, so
checkpoints are directly comparable.


In [ ]:
import os, math, glob, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

SEED = 1234
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- Ablation flags: all 8 combos expressible (all-False invalid) ----
USE_WEIGHTS     = True     # keep the (2,23,128) per-frame STDP weight type-images
USE_INPUT_RATE  = True     # keep the gathered per-frame input_activity planes
USE_HIDDEN_RATE = True     # keep the broadcast per-frame hidden_activity plane
assert USE_WEIGHTS or USE_INPUT_RATE or USE_HIDDEN_RATE, "at least one channel group must be on"
MODE_TAG = f"w{int(USE_WEIGHTS)}_ir{int(USE_INPUT_RATE)}_hr{int(USE_HIDDEN_RATE)}"

# ---- Geometry (matches training/tonotopic_plasticity_bound/train.py's 128-dim,
#      2-types-per-channel architecture, via _fingerprint_core_piecewise.py) ----
N_AUDITORY_CH = 64
N_PER_CHANNEL = 2                      # 128 hidden = 64 channels x 2 (sustained, onset)
N_HIDDEN      = 128
N_INPUT_TYPES = 2                      # sustained / onset (no phase)
R_EXC_CHANNEL = 11
H_OFF         = 2 * R_EXC_CHANNEL + 1  # 23 tonotopic offsets (-11..+11)
IN_CH         = 5                      # 2 weights + 2 input-rate + 1 hidden-rate
EMBED_DIM     = 256
TDNN_C        = 256                    # Bottle2neck channel width (ECAPA-style frame-to-frame stack)
NEUTRAL_FILL  = math.exp(-30)          # zero-variance fill, neutralized by input BatchNorm

# ---- Kaggle paths — point at the piece-wise shard DIRECTORIES from
#      prepare_fingerprints_piecewise.ipynb (NOT the old whole-clip *_fingerprints.npz).
#      Each shard is a DIRECTORY (weights_frames.npy/input_activity_frames.npy/
#      hidden_activity_frames.npy/meta.npz), not a single .npz file -- required for
#      true mmap (see load_split below); the glob patterns match directory names.
#      Shards are uploaded as ONE SEPARATE Kaggle dataset PER SHARD (can't zip all 38
#      together), so all 38 are attached as inputs to this notebook individually and
#      DEV_GLOB/TEST_GLOB wildcard the per-shard dataset-slug level (the level right
#      after the owner) instead of assuming one shared bundling dataset. ----
INPUT_ROOT = "/kaggle/input/datasets/qphulong"
DEV_GLOB  = os.path.join(INPUT_ROOT, "*", "vox2_*_fingerprints_pw")   # train (closed-set)
TEST_GLOB = os.path.join(INPUT_ROOT, "*", "vox1_*_fingerprints_pw")   # eval  (open-set)

# ---- PK sampler / batch ----
P_CLASSES  = 32
K_SAMPLES  = 4
BATCH_SIZE = P_CLASSES * K_SAMPLES     # 128
VAL_BATCH  = 64

# ---- Optim / schedule ----
EPOCHS               = 80
LR                   = 0.08
WEIGHT_DECAY         = 5e-4
WARMUP_EPOCHS        = 3
MARGIN_WARMUP_EPOCHS = 15
AAM_M                = 0.2
AAM_S                = 30
GRAD_CLIP            = 5.0
EARLY_STOP_PATIENCE  = 0               # disabled: always run full EPOCHS
NUM_WORKERS          = 0

# ---- Checkpointing ----
CKPT_DIR  = "/kaggle/working"
LAST_CKPT = os.path.join(CKPT_DIR, f"last_seq_{MODE_TAG}.pt")   # always the SAVE destination, every epoch
BEST_CKPT = os.path.join(CKPT_DIR, f"best_seq_{MODE_TAG}.pt")   # always the SAVE destination, on new best

# Explicit file to RESUME from -- independent of LAST_CKPT/BEST_CKPT above, so it can
# point anywhere (e.g. a checkpoint backed up/reuploaded to a read-only
# /kaggle/input/... dataset if a prior session's /kaggle/working didn't persist).
# Exactly one of the two must be set -- it picks the resume mode:
#   RESUME_FROM_LAST -- true continuation: optimizer/scheduler/scaler state (incl. SGD
#                       momentum buffers) restored too, training picks up with no
#                       discontinuity.
#   RESUME_FROM_BEST -- weights-only rollback to the best epoch so far; opt/sched/
#                       scaler are freshly re-initialized (BEST_CKPT never stored them,
#                       see save_best) and the scheduler is fast-forwarded to match the
#                       resumed epoch count -- momentum restarts from zero. Use this if
#                       recent epochs look worse than an earlier one and you'd rather
#                       continue from the good weights than the degraded ones.
# If the chosen path doesn't exist yet (e.g. the very first run), training just starts
# fresh from epoch 0 -- see the "[resume]" print in train() below.
RESUME_FROM_LAST = LAST_CKPT
RESUME_FROM_BEST = None
assert (RESUME_FROM_LAST is not None) != (RESUME_FROM_BEST is not None), \
    "set exactly one of RESUME_FROM_LAST / RESUME_FROM_BEST, not both/neither"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count()
print(f"device={device}  N_GPU={N_GPU} (single-GPU only)  MODE_TAG={MODE_TAG}")

In [ ]:
# ================= IN_IDX gather (VERIFY vs _fingerprint_core_piecewise.py) =================
def build_in_idx():
    """
    Fixed gather index, shape (2, 23, 128): IN_IDX[t, o, h] = position in
    input_activity(128,) for the input neuron of type t feeding hidden neuron h at
    tonotopic offset (o - R_EXC_CHANNEL). Channel-major layout — must match
    _fingerprint_core_piecewise.py's IN_IDX construction exactly (verified against it
    in the offline smoke test; see check_ann_sequence_smoke.py).
    """
    idx = np.empty((N_INPUT_TYPES, H_OFF, N_HIDDEN), dtype=np.int64)
    for t in range(N_INPUT_TYPES):
        for o in range(H_OFF):
            off = o - R_EXC_CHANNEL                      # -11 .. +11
            for h in range(N_HIDDEN):
                base_ch = h // N_PER_CHANNEL             # hidden -> its own channel (0..63)
                src_ch  = (base_ch + off) % N_AUDITORY_CH
                idx[t, o, h] = src_ch * N_PER_CHANNEL + t    # channel-major
    return idx

IN_IDX = build_in_idx()
assert IN_IDX.shape == (2, 23, 128), IN_IDX.shape
assert 0 <= IN_IDX.min() and IN_IDX.max() < N_HIDDEN
IN_IDX_FLAT = torch.from_numpy(IN_IDX.reshape(-1))       # (2*23*128,)
print("IN_IDX ok:", IN_IDX.shape, "range", int(IN_IDX.min()), int(IN_IDX.max()))


In [ ]:
# ===================== LAZY MMAP LOADING, CSR ragged frames =====================
# At full-corpus scale (~95GB across both splits) the old eager "preallocate one
# buffer, bulk-copy every shard into RAM" approach doesn't fit in Kaggle RAM (~30GB).
# Shards are directories of plain .npy files (see prepare_fingerprints_piecewise.ipynb)
# -- NOT bundled into a .npz, because numpy does not actually lazy-load zip members
# even with mmap_mode='r' (verified: it silently reads the whole array into RAM).
# Plain .npy DOES mmap correctly, so the big frame arrays stay on disk and are only
# paged in for the specific (sample, frame-range) slices a batch actually touches.
#
# Per-sample metadata (person_ids/record_ids/labels/n_pieces, from each shard's tiny
# meta.npz) is loaded eagerly regardless -- negligible size even at full scale.
#
# Deliberately NOT switched to shard-restricted / shard-streamed batching: PKSampler
# below draws its P classes from the FULL class set every batch, unchanged. Restricting
# batches to one shard at a time was already tried and reverted upstream (it starved
# the AAM-softmax head, since one shard doesn't contain every speaker) -- mmap avoids
# that regression entirely, at the cost of some random-read I/O instead of a shard
# streaming rewrite.

def list_shards(pattern):
    """`pattern` matches shard DIRECTORIES (e.g. glob '.../vox2_*_fingerprints_pw')."""
    files = sorted(d for d in glob.glob(pattern) if os.path.isdir(d))
    if not files:
        raise FileNotFoundError(f"no shard directories match {pattern}")
    return files

def scan_counts(dirs):
    """Per shard: (n_samples, n_frames_total) -- touches only the tiny meta.npz."""
    total_samples, total_frames, per_shard = 0, 0, []
    for d in dirs:
        meta = np.load(os.path.join(d, "meta.npz"), allow_pickle=True)
        n_s = len(meta["person_ids"])
        n_f = int(meta["n_pieces"].sum())
        per_shard.append((n_s, n_f))
        total_samples += n_s
        total_frames  += n_f
    return total_samples, total_frames, per_shard

def load_split(dirs, name):
    """Returns (shard_arrays, sample_shard, sample_local_off, PID, RID):
      shard_arrays     : list of (W_mm, IA_mm, HA_mm) np.memmap triples, one per shard
                          (kept open for the FingerprintBank's lifetime -- do not close)
      sample_shard      : (N,) int32, which shard each sample belongs to
      sample_local_off  : (N,2) int64, [start,end) into that shard's memmapped arrays
      PID, RID           : (N,) str -- eager, tiny
    Frame data is NEVER copied into one big buffer -- only touched, per-batch, in
    FingerprintBank.batch()."""
    total_samples, total_frames, per_shard = scan_counts(dirs)
    shard_arrays = []
    sample_shard_parts, sample_off_parts = [], []
    PID_parts, RID_parts = [], []

    for shard_idx, (d, (n_s, n_f)) in enumerate(zip(dirs, per_shard)):
        W_mm  = np.load(os.path.join(d, "weights_frames.npy"),         mmap_mode="r")
        IA_mm = np.load(os.path.join(d, "input_activity_frames.npy"),  mmap_mode="r")
        HA_mm = np.load(os.path.join(d, "hidden_activity_frames.npy"), mmap_mode="r")
        assert W_mm.shape[0] == n_f, f"{d}: weights_frames rows != sum(n_pieces)"
        shard_arrays.append((W_mm, IA_mm, HA_mm))

        meta = np.load(os.path.join(d, "meta.npz"), allow_pickle=True)
        n_pieces = meta["n_pieces"].astype(np.int64)
        local_off = np.concatenate([[0], np.cumsum(n_pieces)])
        sample_shard_parts.append(np.full(n_s, shard_idx, dtype=np.int32))
        sample_off_parts.append(np.stack([local_off[:-1], local_off[1:]], axis=1))
        PID_parts.append(meta["person_ids"].astype(str))
        RID_parts.append(meta["record_ids"].astype(str))

    sample_shard     = np.concatenate(sample_shard_parts)
    sample_local_off = np.concatenate(sample_off_parts, axis=0)
    PID = np.concatenate(PID_parts)
    RID = np.concatenate(RID_parts)
    assert len(sample_shard) == total_samples
    est_gb = total_frames * (2 * H_OFF * N_HIDDEN * 2 + N_HIDDEN * 2 * 2) / 1e9
    print(f"[{name}] {total_samples} samples ({total_frames} frames, "
          f"{total_frames/max(total_samples,1):.1f} avg frames/sample) / {len(dirs)} shards | "
          f"~{est_gb:.2f}GB on disk, memory-mapped (not loaded into RAM)")
    return shard_arrays, sample_shard, sample_local_off, PID, RID

def assert_disjoint(dev_pid, test_pid):
    d, t = set(np.unique(dev_pid)), set(np.unique(test_pid))
    inter = d & t
    assert not inter, f"dev/test speaker overlap: {sorted(inter)[:10]}"
    print(f"[disjoint] dev speakers={len(d)}  test speakers={len(t)}  overlap=0")

def subsample_one_per_session(PID, RID, seed=SEED):
    """<=1 fingerprint (sample, i.e. one wav's whole frame sequence) per
    (person_id, record_id): seeded random draw within each session. Operates only
    on the small per-sample metadata -- shard_arrays are untouched by subsampling,
    the returned indices just select which sample_shard/sample_local_off rows to keep."""
    rng = np.random.default_rng(seed)
    sessions = {}
    for i, (p, r) in enumerate(zip(PID, RID)):
        sessions.setdefault((p, r), []).append(i)
    keep = [rows[rng.integers(len(rows))] if len(rows) > 1 else rows[0]
            for rows in sessions.values()]
    return np.array(sorted(keep), dtype=np.int64)

def encode_labels(pid):
    classes = sorted(set(pid.tolist()))
    c2i = {c: i for i, c in enumerate(classes)}
    y = np.fromiter((c2i[p] for p in pid), dtype=np.int64, count=len(pid))
    return y, len(classes), c2i


In [ ]:
# ===================== BANK / AUGMENT / PK SAMPLER (padded + masked) =====================
class FingerprintAugment:
    """Applied once per BATCH, identically across every frame of every sample in the
    batch. The circular tonotopic roll (dims=-1, the 128-neuron axis) is a property of
    a clip's whole channel-map geometry, not a per-frame one — every frame of one clip
    MUST use the same shift, or the shift becomes physically incoherent frame-to-frame
    and sabotages the TDNN's job of learning cross-frame tonotopic alignment. Gaussian
    noise / amplitude scale are also kept per-batch (not per-frame) so they don't fight
    the extractor's repeat-4..7 weight averaging, which already suppresses per-frame
    noise — independent per-frame noise would mask the temporal structure the sequence
    model is specifically built to exploit. dims=-1 still selects the neuron axis
    regardless of the extra leading (B, L) dims, so this needs no other change."""
    def __call__(self, w, ia, ha):
        if random.random() < 0.5:                            # circular tonotopic roll
            k = random.randint(-2, 2)                         # inclusive both ends
            shift = N_PER_CHANNEL * k
            if shift:
                w  = torch.roll(w,  shifts=shift, dims=-1)
                ia = torch.roll(ia, shifts=shift, dims=-1)
                ha = torch.roll(ha, shifts=shift, dims=-1)
        if random.random() < 0.5:                            # additive gaussian, 1% of own std
            w  = w  + torch.randn_like(w)  * (0.01 * w.std())
            ia = ia + torch.randn_like(ia) * (0.01 * ia.std())
            ha = ha + torch.randn_like(ha) * (0.01 * ha.std())
        if random.random() < 0.5:                            # global amplitude scale
            s = random.uniform(0.95, 1.05)
            w, ia, ha = w * s, ia * s, ha * s
        return w, ia, ha

class FingerprintBank:
    """Big frame arrays stay memory-mapped, one (W_mm,IA_mm,HA_mm) triple per shard
    (see load_split) -- `.batch(idx)` looks up each sample's (shard, local offset),
    reads ONLY that sample's frame slice off disk (np.asarray() on a memmap slice
    materializes just those rows, not the whole shard), pads to the batch's own
    L_max, and returns a boolean mask alongside w/ia/ha/y."""
    def __init__(self, shard_arrays, sample_shard, sample_local_off, y, augment=False):
        self.shard_arrays = shard_arrays                     # list of (W_mm, IA_mm, HA_mm)
        self.sample_shard = sample_shard                     # (N_samples,) int32
        self.sample_local_off = sample_local_off             # (N_samples, 2) int64, [start,end)
        self.y  = torch.from_numpy(np.ascontiguousarray(y)).long()
        self.aug = FingerprintAugment() if augment else None

    def batch(self, idx):
        idx_np = idx.numpy() if torch.is_tensor(idx) else np.asarray(idx)
        shards = self.sample_shard[idx_np]
        offs   = self.sample_local_off[idx_np]
        lengths = offs[:, 1] - offs[:, 0]
        B, L_max = len(idx_np), int(lengths.max())

        w  = torch.zeros(B, L_max, N_INPUT_TYPES, H_OFF, N_HIDDEN, dtype=torch.float32)
        ia = torch.zeros(B, L_max, N_HIDDEN, dtype=torch.float32)
        ha = torch.zeros(B, L_max, N_HIDDEN, dtype=torch.float32)
        mask = torch.zeros(B, L_max, dtype=torch.bool)
        for b in range(B):
            n = int(lengths[b])
            s, e = int(offs[b, 0]), int(offs[b, 1])
            W_mm, IA_mm, HA_mm = self.shard_arrays[int(shards[b])]
            # np.array(..., copy=True) reads only this slice off disk into an owned,
            # writable buffer (the memmap itself is read-only, so torch.from_numpy on
            # a raw slice view would warn about non-writable tensors for no reason).
            w[b, :n]  = torch.from_numpy(np.array(W_mm[s:e],  copy=True)).float()
            ia[b, :n] = torch.from_numpy(np.array(IA_mm[s:e], copy=True)).float()
            ha[b, :n] = torch.from_numpy(np.array(HA_mm[s:e], copy=True)).float()
            mask[b, :n] = True

        y = self.y.index_select(0, torch.as_tensor(idx_np, dtype=torch.long))
        if self.aug is not None:
            w, ia, ha = self.aug(w, ia, ha)
        return w, ia, ha, mask, y

class PKSampler:
    """P classes x K samples per batch, drawn from the FULL class set every batch.
       Sample K with replacement for classes with fewer than K fingerprints."""
    def __init__(self, y, p_classes, k_samples, seed=SEED):
        self.p, self.k = p_classes, k_samples
        self.rng = np.random.default_rng(seed)               # persists across epochs
        self.cls2idx = {}
        for i, c in enumerate(y):
            self.cls2idx.setdefault(int(c), []).append(i)
        self.cls2idx = {c: np.asarray(v) for c, v in self.cls2idx.items()}
        self.classes = np.fromiter(self.cls2idx.keys(), dtype=np.int64)
        self.n = len(y)

    def num_batches(self):
        return max(1, self.n // (self.p * self.k))

    def __iter__(self):
        for _ in range(self.num_batches()):
            chosen = self.rng.choice(self.classes, size=self.p, replace=False)
            batch = []
            for c in chosen:
                pool = self.cls2idx[c]
                batch.extend(self.rng.choice(pool, size=self.k,
                                             replace=len(pool) < self.k).tolist())
            yield torch.as_tensor(batch, dtype=torch.long)

def sequential_batches(n, bs):
    for s in range(0, n, bs):
        yield torch.arange(s, min(s + bs, n), dtype=torch.long)


In [ ]:
# ===================== MODEL =====================
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        st = stride if isinstance(stride, tuple) else (stride, stride)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride=st, padding=1, bias=False)
        self.bn1, self.act1 = nn.BatchNorm2d(cout), nn.PReLU(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        if cin != cout or st != (1, 1):
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride=st, bias=False),
                                       nn.BatchNorm2d(cout))
        else:
            self.short = nn.Identity()
        self.act2 = nn.PReLU(cout)

    def forward(self, x):
        y = self.act1(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return self.act2(y + self.short(x))


class FrameCNNEncoder(nn.Module):
    """Per-frame 2D CNN — identical backbone to the old static FingerprintCNN
    (in_bn -> stem -> 4x strided ResBlock), but ending in a plain global-average-pool
    instead of the old spatial AttentiveStatsPool: that pool was over a flattened
    remnant of two SPATIAL axes (tonotopic-offset x hidden-neuron-index) after conv
    downsampling, not real time, so it's retired here — genuine temporal attentive
    pooling now happens once, over the piece axis, in SequenceFingerprintCNN."""
    def __init__(self, in_ch=5, use_weights=True, use_input_rate=True, use_hidden_rate=True,
                 in_idx_flat=None):
        super().__init__()
        self.use_weights, self.use_input_rate, self.use_hidden_rate = \
            use_weights, use_input_rate, use_hidden_rate
        self.register_buffer("in_idx_flat", in_idx_flat.long(), persistent=False)
        self.in_bn = nn.BatchNorm2d(in_ch)                   # standardizes weights vs rates
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(3, 7), padding=(1, 3), bias=False),
            nn.BatchNorm2d(32), nn.PReLU(32))
        self.block1 = ResBlock(32,  64,  (1, 2))             # (N, 64, 23, 64)
        self.block2 = ResBlock(64,  128, (2, 2))             # (N,128, 12, 32)
        self.block3 = ResBlock(128, 256, (2, 2))             # (N,256,  6, 16)
        self.block4 = ResBlock(256, 256, (2, 2))             # (N,256,  3,  8)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def expand(self, w, ia, ha):
        """Compact (w, ia, ha) -> (N,5,23,128). Flag-off groups -> NEUTRAL_FILL constant.
        N = B*L, i.e. this operates on flattened (batch, piece) frames."""
        N = w.shape[0]
        x = torch.empty(N, IN_CH, H_OFF, N_HIDDEN, device=w.device, dtype=w.dtype)
        x[:, 0:2] = w if self.use_weights else NEUTRAL_FILL
        if self.use_input_rate:
            x[:, 2:4] = ia.index_select(1, self.in_idx_flat).view(N, 2, H_OFF, N_HIDDEN)
        else:
            x[:, 2:4] = NEUTRAL_FILL
        if self.use_hidden_rate:
            x[:, 4:5] = ha.view(N, 1, 1, N_HIDDEN).expand(N, 1, H_OFF, N_HIDDEN)
        else:
            x[:, 4:5] = NEUTRAL_FILL
        return x

    def forward(self, w, ia, ha):
        x = self.in_bn(self.expand(w, ia, ha))
        x = self.stem(x)
        x = self.block4(self.block3(self.block2(self.block1(x))))
        return self.gap(x).flatten(1)                        # (N, 256)


# ── ECAPA-TDNN-style frame-to-frame stack, ported verbatim from
#    ecapa_baseline.ipynb's SEModule/Bottle2neck (generic dilated-Conv1d blocks,
#    agnostic to what the 1D axis represents — here the piece/time axis) ─────────
class SEModule(nn.Module):
    def __init__(self, channels, bottleneck=128):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, bottleneck, kernel_size=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(bottleneck, channels, kernel_size=1, padding=0),
            nn.Sigmoid())

    def forward(self, x):
        return x * self.se(x)


class Bottle2neck(nn.Module):
    def __init__(self, inplanes, planes, kernel_size=None, dilation=None, scale=8):
        super().__init__()
        width = int(math.floor(planes / scale))
        self.conv1 = nn.Conv1d(inplanes, width * scale, kernel_size=1)
        self.bn1   = nn.BatchNorm1d(width * scale)
        self.nums  = scale - 1
        num_pad = math.floor(kernel_size / 2) * dilation
        self.convs = nn.ModuleList([
            nn.Conv1d(width, width, kernel_size=kernel_size, dilation=dilation,
                      padding=num_pad) for _ in range(self.nums)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(width) for _ in range(self.nums)])
        self.conv3 = nn.Conv1d(width * scale, planes, kernel_size=1)
        self.bn3   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU()
        self.width = width
        self.se    = SEModule(planes)

    def forward(self, x):
        residual = x
        out = self.bn1(self.relu(self.conv1(x)))
        spx = torch.split(out, self.width, 1)
        for i in range(self.nums):
            sp = spx[i] if i == 0 else sp + spx[i]
            sp = self.bns[i](self.relu(self.convs[i](sp)))
            out = sp if i == 0 else torch.cat((out, sp), 1)
        out = torch.cat((out, spx[self.nums]), 1)
        out = self.bn3(self.relu(self.conv3(out)))
        return self.se(out) + residual


def _mask_zero(x, mask):
    """Zero out padded (mask==False) timesteps of x:(B,C,L) given mask:(B,L). Keeps
    garbage padding from ever entering a conv's receptive field beyond the first op."""
    return x * mask.unsqueeze(1).to(x.dtype)


class MaskedAttentiveStatsPool(nn.Module):
    """Same attention formula as ecapa_baseline.ipynb's ECAPA_TDNN.forward attention
    block, adapted for a padding mask: mu/sigma for the `global_x` context are
    computed over VALID timesteps only (not a plain unmasked mean/var), and the
    attention logits are -inf-filled at padded positions before the softmax over L
    (same "masked_fill before softmax" idiom already used by this codebase's
    retrieval_metrics), so padded positions get exactly 0 attention weight."""
    def __init__(self, in_dim, att_dim=256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(in_dim * 3, att_dim, kernel_size=1), nn.ReLU(), nn.BatchNorm1d(att_dim),
            nn.Tanh(), nn.Conv1d(att_dim, in_dim, kernel_size=1))

    def forward(self, x, mask):                              # x:(B,C,L)  mask:(B,L) bool
        m = mask.unsqueeze(1).float()                         # (B,1,L)
        lengths = m.sum(dim=2, keepdim=True).clamp_min(1.0)   # (B,1,1)
        mu  = (x * m).sum(dim=2, keepdim=True) / lengths
        var = (((x - mu) ** 2) * m).sum(dim=2, keepdim=True) / lengths
        sigma = torch.sqrt(var.clamp_min(1e-4))
        t = x.size(-1)
        global_x = torch.cat([x, mu.expand(-1, -1, t), sigma.expand(-1, -1, t)], dim=1)
        w = self.attention(global_x)
        w = w.masked_fill(~mask.unsqueeze(1), float("-inf"))
        w = torch.softmax(w, dim=2)
        mu_p = torch.sum(x * w, dim=2)
        sg_p = torch.sqrt((torch.sum((x ** 2) * w, dim=2) - mu_p ** 2).clamp_min(1e-4))
        return torch.cat([mu_p, sg_p], dim=1)                 # (B, 2*C)


class SequenceFingerprintCNN(nn.Module):
    """Frame-to-frame speaker embedding: per-frame 2D CNN -> masked TDNN
    (Bottle2neck x3, dilation 2/3/4) over the piece axis -> masked attentive
    statistics pooling -> 256-d embedding. Reads genuine frame-to-frame
    relationships (the TDNN's dilated convs mix neighboring pieces), unlike the
    old static model which pooled a flattened spatial remnant."""
    def __init__(self, in_ch=5, embed_dim=256, tdnn_c=256,
                 use_weights=True, use_input_rate=True, use_hidden_rate=True,
                 in_idx_flat=None):
        super().__init__()
        self.frame_encoder = FrameCNNEncoder(in_ch, use_weights, use_input_rate,
                                             use_hidden_rate, in_idx_flat)
        self.tdnn_in = nn.Sequential(
            nn.Conv1d(256, tdnn_c, kernel_size=5, padding=2), nn.ReLU(), nn.BatchNorm1d(tdnn_c))
        self.layer1 = Bottle2neck(tdnn_c, tdnn_c, kernel_size=3, dilation=2, scale=8)
        self.layer2 = Bottle2neck(tdnn_c, tdnn_c, kernel_size=3, dilation=3, scale=8)
        self.layer3 = Bottle2neck(tdnn_c, tdnn_c, kernel_size=3, dilation=4, scale=8)
        self.layer4 = nn.Conv1d(3 * tdnn_c, 1536, kernel_size=1)
        self.pool = MaskedAttentiveStatsPool(1536)
        self.bn5 = nn.BatchNorm1d(3072)
        self.fc6 = nn.Linear(3072, embed_dim, bias=False)
        self.bn6 = nn.BatchNorm1d(embed_dim)
        self._init_weights()

    def _init_weights(self):
        for m in (self.tdnn_in, self.layer4, self.fc6):
            for mm in m.modules() if hasattr(m, "modules") else [m]:
                if isinstance(mm, (nn.Conv1d, nn.Linear)):
                    nn.init.kaiming_normal_(mm.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, w, ia, ha, mask):
        # w:(B,L,2,23,128)  ia/ha:(B,L,128)  mask:(B,L)
        B, L = w.shape[0], w.shape[1]
        w_flat  = w.reshape(B * L, *w.shape[2:])
        ia_flat = ia.reshape(B * L, ia.shape[-1])
        ha_flat = ha.reshape(B * L, ha.shape[-1])
        f = self.frame_encoder(w_flat, ia_flat, ha_flat)      # (B*L, 256)
        f = f.view(B, L, 256).transpose(1, 2)                 # (B, 256, L)
        f = _mask_zero(f, mask)

        x = _mask_zero(self.tdnn_in(f), mask)
        x1 = _mask_zero(self.layer1(x), mask)
        x2 = _mask_zero(self.layer2(x + x1), mask)
        x3 = _mask_zero(self.layer3(x + x1 + x2), mask)
        x  = _mask_zero(torch.relu(self.layer4(torch.cat((x1, x2, x3), dim=1))), mask)

        pooled = self.pool(x, mask)                           # (B, 3072)
        emb = self.bn6(self.fc6(self.bn5(pooled)))
        return emb                                            # (B,256), L2-normalize at eval


In [ ]:
# ===================== AAMSoftmax (ArcFace) — reused unchanged =====================
class AAMSoftmax(nn.Module):
    def __init__(self, embed_dim, num_classes, m=0.2, s=30):
        super().__init__()
        self.s = s
        self.W = nn.Parameter(torch.empty(num_classes, embed_dim))
        nn.init.xavier_normal_(self.W)
        self.set_margin(m)

    def set_margin(self, m):
        self.m = m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)                      # easy-margin threshold
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cos = F.linear(F.normalize(emb), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt((1 - cos ** 2).clamp_min(1e-9))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)  # easy-margin guard
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi + (1 - one_hot) * cos) * self.s
        return F.cross_entropy(logits, labels)


In [ ]:
# ===================== OPTIMIZER + SCHEDULE — reused unchanged =====================
def build_optimizer(model, aam, lr, wd):
    """Weight decay on conv/linear weights (incl. AAM W) only; not BN/PReLU/biases."""
    decay, no_decay = [], []
    for module in (model, aam):
        for name, p in module.named_parameters():
            if not p.requires_grad:
                continue
            (no_decay if p.ndim <= 1 or name.endswith(".bias") else decay).append(p)
    return SGD([{"params": decay,    "weight_decay": wd},
                {"params": no_decay, "weight_decay": 0.0}],
               lr=lr, momentum=0.9, nesterov=True)

def build_scheduler(opt, epochs, warmup):
    warm = LinearLR(opt, start_factor=0.1, total_iters=warmup)
    cos  = CosineAnnealingLR(opt, T_max=epochs - warmup, eta_min=1e-5)
    return SequentialLR(opt, [warm, cos], milestones=[warmup])


In [ ]:
# ===================== EVAL: session-free vs leaky, block-wise =====================
# retrieval_metrics / _eer_from_hist / evaluate are reused UNCHANGED from
# ann_backend.ipynb — they consume only the final (N,256) embedding matrix, agnostic
# to how it was produced. Only extract_embeddings changes (threads `mask` through).
@torch.no_grad()
def extract_embeddings(net, bank, n):
    net.eval()
    out = torch.empty(n, EMBED_DIM, dtype=torch.float32)
    pos = 0
    for idx in sequential_batches(n, VAL_BATCH):
        w, ia, ha, mask, _ = bank.batch(idx)
        with torch.autocast("cuda", dtype=torch.float16):
            e = net(w.to(device), ia.to(device), ha.to(device), mask.to(device))
        e = F.normalize(e.float(), dim=1).cpu()
        out[pos:pos + e.shape[0]] = e
        pos += e.shape[0]
    return out

def _eer_from_hist(gen_hist, imp_hist):
    g  = gen_hist / max(gen_hist.sum(), 1)
    im = imp_hist / max(imp_hist.sum(), 1)
    frr = np.cumsum(g)               # genuine in bins <= thr -> rejected
    far = 1.0 - np.cumsum(im)        # impostor in bins  > thr -> accepted
    k = int(np.argmin(np.abs(frr - far)))
    return float((frr[k] + far[k]) / 2)

@torch.no_grad()
def retrieval_metrics(embs, labels, record_ids, session_free, block=512, n_bins=20000):
    N = embs.shape[0]
    E   = embs.to(device)
    lab = torch.as_tensor(labels, device=device)
    rec = torch.as_tensor(np.unique(record_ids, return_inverse=True)[1], device=device)
    gen_hist = np.zeros(n_bins); imp_hist = np.zeros(n_bins)
    r1 = r5 = 0; ap_sum = 0.0; valid = 0
    for s in range(0, N, block):
        e    = E[s:s + block]
        sims = e @ E.t()                                     # (b, N)
        b    = sims.shape[0]
        rows = torch.arange(s, s + b, device=device)
        if session_free:
            excl = rec[rows][:, None] == rec[None, :]        # same recording (incl. self)
        else:
            excl = rows[:, None] == torch.arange(N, device=device)[None, :]   # self only
        sims = sims.masked_fill(excl, -2.0)
        same = (lab[rows][:, None] == lab[None, :]) & (~excl)
        order = torch.argsort(sims, dim=1, descending=True)
        same_sorted = torch.gather(same, 1, order)
        r1 += same_sorted[:, 0].sum().item()
        r5 += same_sorted[:, :5].any(dim=1).sum().item()
        rel = same.sum(dim=1)
        for i in range(b):
            rc = int(rel[i])
            if rc == 0:
                continue
            hit_ranks = same_sorted[i].nonzero(as_tuple=False).squeeze(1).float() + 1
            precs = torch.arange(1, rc + 1, device=device).float() / hit_ranks
            ap_sum += float(precs.sum() / rc); valid += 1
        gen_hist += torch.histc(sims[same],              bins=n_bins, min=-1, max=1).cpu().numpy()
        imp_hist += torch.histc(sims[(~same) & (~excl)], bins=n_bins, min=-1, max=1).cpu().numpy()
    return dict(rank1=r1 / max(valid, 1), rank5=r5 / max(valid, 1),
                mAP=ap_sum / max(valid, 1), eer=_eer_from_hist(gen_hist, imp_hist))

def evaluate(net, val_bank, labels, records, has_dup):
    embs = extract_embeddings(net, val_bank, len(labels))
    sf = retrieval_metrics(embs, labels, records, session_free=True)
    lk = retrieval_metrics(embs, labels, records, session_free=False) if has_dup else sf
    return sf, lk


In [ ]:
def mem_report():
    rss = rss_gb()
    if torch.cuda.is_available():
        return (f"RSS={rss:.2f}GB "
                f"gpu_resv={torch.cuda.memory_reserved()/1e9:.2f} "
                f"gpu_alloc={torch.cuda.memory_allocated()/1e9:.2f}")
    return f"RSS={rss:.2f}GB"

# ===================== TRAINING =====================
def rss_gb():
    with open("/proc/self/statm") as f:
        pages = int(f.read().split()[1])
    return pages * os.sysconf("SC_PAGE_SIZE") / 1e9

# _use_new_zipfile_serialization=False: PyTorch's default .pt format (>=1.6) is a zip
# archive, and Kaggle auto-extracts ANY zip-shaped file it finds -- recursively, even
# nested inside another zip you upload as a wrapper. That silently shreds a normal .pt
# checkpoint into loose data.pkl/data/* files that torch.load can no longer open. The
# legacy (pre-1.6) format is a flat pickle stream with no zip magic bytes, so Kaggle
# has nothing to extract -- it survives being zipped, uploaded, and re-extracted intact.
def save_last(path, net, aam, opt, sched, scaler, history, epoch, best_eer):
    torch.save(dict(mode_tag=MODE_TAG, epoch=epoch, best_eer=best_eer, history=history,
                    model=net.state_dict(), aam=aam.state_dict(),
                    opt=opt.state_dict(), sched=sched.state_dict(),
                    scaler=scaler.state_dict()), path, _use_new_zipfile_serialization=False)

def save_best(path, net, aam, epoch, metrics):
    torch.save(dict(mode_tag=MODE_TAG, epoch=epoch, metrics=metrics,
                    model=net.state_dict(), aam=aam.state_dict()), path,
               _use_new_zipfile_serialization=False)

def banner(where):
    print(f"===== RUN CONFIG ({where}) =====")
    print(f"  MODE_TAG={MODE_TAG}  W={USE_WEIGHTS} IR={USE_INPUT_RATE} HR={USE_HIDDEN_RATE}")
    print(f"  EPOCHS={EPOCHS}  EARLY_STOP_PATIENCE={EARLY_STOP_PATIENCE} (0=disabled)")
    print(f"  N_GPU={N_GPU} (single-GPU)  AMP=fp16 autocast + GradScaler")
    print("=================================")

def train():
    # ---- data (shard_arrays: per-shard memmap triples, never copied into RAM;
    #      sample_shard/sample_local_off: which shard + which frame range per sample) ----
    shards_d, sshard_d, soff_d, PIDd, RIDd = load_split(list_shards(DEV_GLOB),  "dev")
    shards_t, sshard_t, soff_t, PIDt, RIDt = load_split(list_shards(TEST_GLOB), "test")
    assert_disjoint(PIDd, PIDt)

    # Subsampling selects a SUBSET of sample-level rows only — shard_arrays (the
    # actual memmapped frame data) are never touched/rebuilt by this.
    kd = subsample_one_per_session(PIDd, RIDd)
    kt = subsample_one_per_session(PIDt, RIDt)
    sshard_d, soff_d, PIDd, RIDd = sshard_d[kd], soff_d[kd], PIDd[kd], RIDd[kd]
    sshard_t, soff_t, PIDt, RIDt = sshard_t[kt], soff_t[kt], PIDt[kt], RIDt[kt]
    print(f"[subsample] dev {len(kd)}  test {len(kt)} (one fingerprint movie per session)")

    yd, num_classes, _ = encode_labels(PIDd)
    yt, _, _           = encode_labels(PIDt)                 # test labels: retrieval grouping only

    HAS_DUP = len(np.unique(RIDt)) < len(RIDt)
    print(f"[eval] HAS_DUP_SESSIONS={HAS_DUP} "
          f"(session-free vs leaky {'differ' if HAS_DUP else 'collapse -> single pass'})")

    # PKSampler still draws its P classes from the FULL dev class set every batch
    # (unchanged) -- shard membership is invisible to sampling, only affects which
    # memmap a sample's frames get read from in FingerprintBank.batch().
    train_bank = FingerprintBank(shards_d, sshard_d, soff_d, yd, augment=True)
    val_bank   = FingerprintBank(shards_t, sshard_t, soff_t, yt, augment=False)
    sampler    = PKSampler(yd, P_CLASSES, K_SAMPLES)

    # ---- model / loss / optim ----
    net = SequenceFingerprintCNN(IN_CH, EMBED_DIM, TDNN_C,
                                 USE_WEIGHTS, USE_INPUT_RATE, USE_HIDDEN_RATE,
                                 IN_IDX_FLAT).to(device)
    aam = AAMSoftmax(EMBED_DIM, num_classes, m=AAM_M, s=AAM_S).to(device)
    opt    = build_optimizer(net, aam, LR, WEIGHT_DECAY)
    sched  = build_scheduler(opt, EPOCHS, WARMUP_EPOCHS)
    scaler = torch.cuda.amp.GradScaler()

    history, best_eer, start_epoch = [], float("inf"), 0
    banner("after setup")

    # ---- sanity forward (eval + no_grad so BN running stats stay clean) ----
    net.eval()
    with torch.no_grad():
        w0, ia0, ha0, mask0, _ = train_bank.batch(torch.arange(min(8, len(yd))))
        _ = net(w0.to(device), ia0.to(device), ha0.to(device), mask0.to(device))
    net.train()
    print("[sanity] forward pass OK")

    # ---- resume ----
    if RESUME_FROM_LAST is not None:
        resume_path, resume_mode = RESUME_FROM_LAST, "last"
    else:
        resume_path, resume_mode = RESUME_FROM_BEST, "best"
    if os.path.exists(resume_path):
        ck = torch.load(resume_path, map_location=device)
        if ck.get("mode_tag") == MODE_TAG:
            net.load_state_dict(ck["model"]); aam.load_state_dict(ck["aam"])
            start_epoch = ck["epoch"] + 1
            if resume_mode == "last":
                opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
                scaler.load_state_dict(ck["scaler"])
                history, best_eer = ck["history"], ck["best_eer"]
                print(f"[resume:last] {resume_path} -> epoch {start_epoch}  best_eer={best_eer:.4f}")
            else:
                # BEST_CKPT has no opt/sched/scaler (see save_best) -- opt/sched/scaler
                # stay freshly built above; fast-forward the scheduler so LR/warmup
                # position matches start_epoch (both sub-schedulers are step-count-only,
                # so this reproduces the same LR trajectory training would have hit).
                for _ in range(start_epoch):
                    sched.step()
                history, best_eer = [], ck["metrics"]["eer"]
                print(f"[resume:best] {resume_path} -> epoch {start_epoch}  best_eer={best_eer:.4f} "
                      f"(optimizer/scheduler state reset -- momentum restarts fresh)")
        else:
            print(f"[resume] MODE_TAG mismatch ({ck.get('mode_tag')} != {MODE_TAG}); fresh start")
    else:
        print(f"[resume] no checkpoint at {resume_path}; fresh start")

    banner("before training loop")

    for epoch in range(start_epoch, EPOCHS):
        aam.set_margin(AAM_M * min(1.0, epoch / max(1, MARGIN_WARMUP_EPOCHS)))  # margin warmup
        net.train()
        t0, running, nb = time.time(), 0.0, 0

        for idx in sampler:
            w, ia, ha, mask, y = train_bank.batch(idx)
            w, ia, ha, mask, y = w.to(device), ia.to(device), ha.to(device), mask.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                emb = net(w, ia, ha, mask)
            loss = aam(emb.float(), y)                        # ArcFace in fp32
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(net.parameters()) + list(aam.parameters()),
                                           GRAD_CLIP)
            scaler.step(opt); scaler.update()
            running += loss.item(); nb += 1
        sched.step()

        sf, lk = evaluate(net, val_bank, yt, RIDt, HAS_DUP)

        improved = sf["eer"] < best_eer
        if improved:
            best_eer = sf["eer"]
        history.append(dict(epoch=epoch, loss=running / max(nb, 1), sf=sf, leaky=lk,
                            rss=rss_gb(), lr=opt.param_groups[0]["lr"]))

        print(f"[e{epoch:02d}] loss={history[-1]['loss']:.4f} "
              f"m={aam.m:.3f} lr={history[-1]['lr']:.2e} | "
              f"SF eer={sf['eer']:.4f} r1={sf['rank1']:.3f} r5={sf['rank5']:.3f} mAP={sf['mAP']:.3f} | "
              f"LK eer={lk['eer']:.4f} r1={lk['rank1']:.3f} | "
              f"{mem_report()} {'*best' if improved else ''} "
              f"({time.time() - t0:.0f}s)")

        save_last(LAST_CKPT, net, aam, opt, sched, scaler, history, epoch, best_eer)
        if improved:
            save_best(BEST_CKPT, net, aam, epoch, sf)

    print(f"[done] best session-free EER={best_eer:.4f}  ->  {BEST_CKPT}")
    return history


In [ ]:
# ===================== RUN =====================
history = train()
